In [ ]:
!pip install -q insightface==0.7.3 onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 13.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 71.9 MB/s eta 0:00:00


In [ ]:
!pip install onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 6.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/face_recognition_demo")
KNOWN_FACES_DIR = PROJECT_DIR / "known_faces"
TEST_IMAGES_DIR = PROJECT_DIR / "test_images"
DB_PATH = PROJECT_DIR / "face_db.pkl"

In [ ]:
import cv2
import numpy as np
import pickle
from pathlib import Path

In [ ]:
from insightface.app import FaceAnalysis

In [ ]:
app = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 77076.62KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [ ]:
import os
import numpy as np
import cv2
import pickle

def l2_norm(x):
    return x / np.linalg.norm(x)

def build_database():
    db = {}

    for person_dir in KNOWN_FACES_DIR.iterdir():
        if not person_dir.is_dir():
            continue

        name = person_dir.name
        embeddings = []

        for img_path in person_dir.glob("*"):
            img = cv2.imread(str(img_path))
            if img is None:
                continue

            faces = app.get(img)
            if len(faces) == 0:
                continue

            face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
            emb = l2_norm(face.embedding)
            embeddings.append(emb)

        if len(embeddings) > 0:
            db[name] = np.mean(embeddings, axis=0)
            print(f"[OK] {name} -> {len(embeddings)} images")

    with open(DB_PATH, "wb") as f:
        pickle.dump(db, f)

    print("Database selesai. Total:", len(db))

    return db

In [ ]:
db = build_database()

[OK] Minji -> 3 images
[OK] Kroos -> 3 images
[OK] Radit -> 3 images
Database selesai. Total: 3


In [ ]:
def recognize(img_path):
    img = cv2.imread(img_path)
    faces = app.get(img)

    results = []

    for face in faces:
        emb = l2_norm(face.embedding)

        best_name = "Unknown"
        best_score = -1

        for name, ref in db.items():
            score = np.dot(emb, ref)
            if score > best_score:
                best_score = score
                best_name = name

        results.append((best_name, best_score))

    return results

In [ ]:
img_path = str(TEST_IMAGES_DIR / "test_1.jpg")
print(recognize(img_path))

[('Kroos', np.float32(0.73895717))]


In [ ]:
from google.colab.patches import cv2_imshow

def recognize_and_draw(img_path):
    img = cv2.imread(img_path)

    if img is None:
        print("Gambar tidak terbaca")
        return

    faces = app.get(img)

    results = []

    for face in faces:
        emb = face.embedding / np.linalg.norm(face.embedding)

        best_name = "Unknown"
        best_score = -1

        for name, ref in db.items():
            score = np.dot(emb, ref)

            if score > best_score:
                best_score = score
                best_name = name

        # threshold penting
        if best_score < 0.45:
            label = f"Unknown ({best_score:.2f})"
            color = (0, 0, 255)
        else:
            label = f"{best_name} ({best_score:.2f})"
            color = (0, 255, 0)

        x1, y1, x2, y2 = map(int, face.bbox)

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        results.append((best_name, best_score))

    cv2_imshow(img)
    return results

In [ ]:
img_path = str(TEST_IMAGES_DIR / "test_1.jpg")
recognize_and_draw(img_path)